<a href="https://colab.research.google.com/github/kashidkshitija-gif/Agile-Sprint-Prediction/blob/main/AgileProject_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas scikit-learn joblib gradio matplotlib
import pandas as pd

data = pd.read_csv("agile_sprint_data.csv")

data.head()

,sprint_id,team_size,sprint_length_days,story_points_committed,story_points_completed,velocity,backlog_changes,defects_reported,defects_reopened,automation_coverage,build_failures,avg_cycle_time,sprint_success
0,S001,8,14,45,24,48,7,9,0,76,1,2.8,0
1,S002,9,14,48,55,32,6,8,0,82,1,2.6,1
2,S003,7,14,33,53,54,8,2,2,45,5,3.7,0
3,S004,9,10,32,50,25,6,14,3,78,1,4.6,0
4,S005,9,14,46,29,37,2,3,1,85,4,6.4,0


In [4]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Split
X = data.drop(["sprint_success", "sprint_id"], axis=1) # Drop sprint_id as it's a non-numeric identifier
y = data["sprint_success"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# Accuracy
y_pred = model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, y_pred))

# Save model
joblib.dump(model, "sprint_model.pkl")

Model Accuracy: 0.925


['sprint_model.pkl']

In [5]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
import joblib
import io
import base64

model = joblib.load("sprint_model.pkl")

def plot_chart(prob):
    plt.figure()
    plt.bar(["Failure", "Success"], [1-prob, prob])
    plt.ylim(0,1)

    buf = io.BytesIO()
    plt.savefig(buf, format="png")
    buf.seek(0)

    img = base64.b64encode(buf.read()).decode()
    plt.close()

    return f'<img src="data:image/png;base64,{img}"/>'

def predict(team_size, sprint_length, committed, completed,
            velocity, backlog_changes, defects_reported,
            defects_reopened, automation, build_failures, cycle_time):

    input_df = pd.DataFrame([[team_size, sprint_length, committed, completed,
                              velocity, backlog_changes, defects_reported,
                              defects_reopened, automation, build_failures,
                              cycle_time]],
                            columns=model.feature_names_in_)

    prediction = model.predict(input_df)[0]
    prob = model.predict_proba(input_df)[0][1]

    chart = plot_chart(prob)

    if prediction == 1:
        status = " Sprint Likely Successful"
        color = "green"
    else:
        status = " Sprint At Risk"
        color = "red"

    html = f"""
    <div style="text-align:center">
        <h2 style='color:{color}'>{status}</h2>
        <h3>Success Probability: {prob*100:.2f}%</h3>
        {chart}
    </div>
    """

    return html

with gr.Blocks() as demo:
    gr.Markdown("#  Agile Sprint Success Predictor")

    team_size = gr.Number(label="Team Size")
    sprint_length = gr.Number(label="Sprint Length")
    committed = gr.Number(label="Committed Story Points")
    completed = gr.Number(label="Completed Story Points")
    velocity = gr.Number(label="Velocity")
    backlog_changes = gr.Number(label="Backlog Changes")
    defects_reported = gr.Number(label="Defects Reported")
    defects_reopened = gr.Number(label="Defects Reopened")
    automation = gr.Number(label="Automation Coverage")
    build_failures = gr.Number(label="Build Failures")
    cycle_time = gr.Number(label="Cycle Time")

    output = gr.HTML()

    btn = gr.Button("Predict Sprint")

    btn.click(
        predict,
        inputs=[team_size, sprint_length, committed, completed,
                velocity, backlog_changes, defects_reported,
                defects_reopened, automation, build_failures, cycle_time],
        outputs=output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b8b070efba241991c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
